In [1]:
!pip install -q transformers datasets timm accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 66.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!nvidia-smi

Fri Mar  6 04:39:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!cp -r /content/drive/MyDrive/Agromate/PlantVillage_1000_SPLIT /content/

In [5]:
import os

DATASET_ROOT = "/content/PlantVillage_1000_SPLIT"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")

classes = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(classes)

print("Total Classes:", NUM_CLASSES)

Total Classes: 38


In [6]:
from transformers import DeiTImageProcessor

processor = DeiTImageProcessor.from_pretrained(
    "facebook/deit-tiny-patch16-224"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [7]:
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    )
])

In [8]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=val_transform)

In [13]:
from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [14]:
import torch
from transformers import DeiTForImageClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DeiTForImageClassification.from_pretrained(
    "facebook/deit-tiny-patch16-224",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True   # 🔥 ADD THIS
)

model.to(device)

You are using a model of type vit to instantiate a model of type deit. This is not supported for all configurations of models and can yield errors.


Loading weights:   0%|          | 0/2 [00:00<?, ?it/s]

DeiTForImageClassification LOAD REPORT from: facebook/deit-tiny-patch16-224
Key                                                          | Status     |                                                                                         
-------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
vit.encoder.layer.{0...11}.output.dense.bias                 | UNEXPECTED |                                                                                         
vit.encoder.layer.{0...11}.layernorm_before.bias             | UNEXPECTED |                                                                                         
vit.encoder.layer.{0...11}.attention.output.dense.weight     | UNEXPECTED |                                                                                         
vit.encoder.layer.{0...11}.layernorm_after.weight            | UNEXPECTED |                        

DeiTForImageClassification(
  (deit): DeiTModel(
    (embeddings): DeiTEmbeddings(
      (patch_embeddings): DeiTPatchEmbeddings(
        (projection): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DeiTEncoder(
      (layer): ModuleList(
        (0-11): 12 x DeiTLayer(
          (attention): DeiTAttention(
            (attention): DeiTSelfAttention(
              (query): Linear(in_features=192, out_features=192, bias=True)
              (key): Linear(in_features=192, out_features=192, bias=True)
              (value): Linear(in_features=192, out_features=192, bias=True)
            )
            (output): DeiTSelfOutput(
              (dense): Linear(in_features=192, out_features=192, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): DeiTIntermediate(
            (dense): Linear(in_features=192, out_features=768, bias=True)
           

In [15]:
import torch.nn as nn
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=3e-5)
criterion = nn.CrossEntropyLoss()

In [16]:
EPOCHS = 20
patience = 3

best_val_acc = 0
counter = 0

for epoch in range(EPOCHS):

    # ===== TRAINING =====
    model.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images).logits
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = 100 * train_correct / train_total


    # ===== VALIDATION =====
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images).logits
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(f"\nEpoch {epoch+1}")
    print(f"Train Acc: {train_acc:.2f}%")
    print(f"Val Acc: {val_acc:.2f}%")


    # ===== EARLY STOPPING PART =====

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0

        # Save best model
        torch.save(model.state_dict(),
                   "/content/drive/MyDrive/Agromate/best_deit_model1.pth")

        print("✅ Best model saved")

    else:
        counter += 1
        print(f"No improvement for {counter} epoch(s)")

        if counter >= patience:
            print("⛔ Early stopping triggered")
            break


Epoch 1
Train Acc: 52.53%
Val Acc: 65.86%
✅ Best model saved

Epoch 2
Train Acc: 74.00%
Val Acc: 74.03%
✅ Best model saved

Epoch 3
Train Acc: 82.24%
Val Acc: 84.19%
✅ Best model saved

Epoch 4
Train Acc: 86.37%
Val Acc: 84.68%
✅ Best model saved

Epoch 5
Train Acc: 89.07%
Val Acc: 87.81%
✅ Best model saved

Epoch 6
Train Acc: 90.75%
Val Acc: 89.82%
✅ Best model saved

Epoch 7
Train Acc: 92.32%
Val Acc: 91.51%
✅ Best model saved

Epoch 8
Train Acc: 93.37%
Val Acc: 91.36%
No improvement for 1 epoch(s)

Epoch 9
Train Acc: 93.86%
Val Acc: 92.10%
✅ Best model saved

Epoch 10
Train Acc: 94.32%
Val Acc: 93.62%
✅ Best model saved

Epoch 11
Train Acc: 94.97%
Val Acc: 93.53%
No improvement for 1 epoch(s)

Epoch 12
Train Acc: 95.63%
Val Acc: 93.70%
✅ Best model saved

Epoch 13
Train Acc: 95.74%
Val Acc: 94.00%
✅ Best model saved

Epoch 14
Train Acc: 96.16%
Val Acc: 93.68%
No improvement for 1 epoch(s)

Epoch 15
Train Acc: 96.31%
Val Acc: 95.56%
✅ Best model saved

Epoch 16
Train Acc: 96.86%
Val

In [17]:
SAVE_PATH = "/content/drive/MyDrive/Agromate/deit_plant_disease_model_small1"

model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)

print("✅ Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model Saved Successfully!


In [18]:
import torch
from transformers import DeiTForImageClassification, DeiTImageProcessor
from PIL import Image
import os

# 🔹 Path where best model was saved
MODEL_PATH = "/content/drive/MyDrive/Agromate"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 🔹 Load processor
processor = DeiTImageProcessor.from_pretrained(
    "facebook/deit-tiny-patch16-224"
)

# 🔹 Load model
NUM_CLASSES = 38  # change if different
model = DeiTForImageClassification.from_pretrained(
    "facebook/deit-tiny-patch16-224",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)

model.load_state_dict(torch.load(
    "/content/drive/MyDrive/Agromate/best_deit_model1.pth",
    map_location=device
))

model.to(device)
model.eval()

print("✅ Model Loaded Successfully!")

You are using a model of type vit to instantiate a model of type deit. This is not supported for all configurations of models and can yield errors.


Loading weights:   0%|          | 0/2 [00:00<?, ?it/s]

DeiTForImageClassification LOAD REPORT from: facebook/deit-tiny-patch16-224
Key                                                          | Status     |                                                                                         
-------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
vit.encoder.layer.{0...11}.output.dense.bias                 | UNEXPECTED |                                                                                         
vit.encoder.layer.{0...11}.layernorm_before.bias             | UNEXPECTED |                                                                                         
vit.encoder.layer.{0...11}.attention.output.dense.weight     | UNEXPECTED |                                                                                         
vit.encoder.layer.{0...11}.layernorm_after.weight            | UNEXPECTED |                        

✅ Model Loaded Successfully!


In [20]:
TRAIN_DIR = "/content/PlantVillage_1000_SPLIT/train"
class_names = sorted(os.listdir(TRAIN_DIR))

print("Classes Loaded:", len(class_names))

Classes Loaded: 38


In [21]:
def predict_image(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_idx = torch.argmax(logits, dim=1).item()

    predicted_class = class_names[predicted_class_idx]

    return predicted_class

In [27]:
image_path = "/content/PlantVillage_1000_SPLIT/test/Apple___Cedar_apple_rust/bcc248dc-0c65-46f0-b80e-bac2b8db5045___FREC_C.Rust 0013.JPG"

prediction = predict_image(image_path)

print("🌿 Predicted Class:", prediction)

🌿 Predicted Class: Apple___Cedar_apple_rust


# Task
Modify cell `TKxW3ralChoW` to clear the `/content/drive` directory by removing its contents if it exists and is not empty, and then recreate it. Then, execute the modified cell to mount Google Drive.

## Modify cell TKxW3ralChoW to clear the mount point

### Subtask:
Add code to delete the `/content/drive` directory if it exists and is not empty, then recreate it, before calling `drive.mount`.


**Reasoning**:
The subtask requires modifying the `drive.mount` cell to handle pre-existing files at the mount point. I will add imports for `os` and `shutil`, then conditionally remove and recreate the `/content/drive` directory before attempting to mount it, as per the instructions.



In [ ]:
import os
import shutil
from google.colab import drive

mount_point = '/content/drive'

# Check if the mount point exists and remove it if it's not empty
if os.path.exists(mount_point) and os.listdir(mount_point):
    print(f"Removing existing mount point: {mount_point}")
    shutil.rmtree(mount_point)

# Create an empty directory for the mount point
os.makedirs(mount_point, exist_ok=True)

drive.mount(mount_point, force_remount=True)

Removing existing mount point: /content/drive
Mounted at /content/drive
